# 06 — Results
## Final summary: what the data says, what it means, what remains open

This notebook loads the saved diagnostic output and constructs the final summary table.
It reads `output/diagnostic_summary.json` (from `bullet_engine.py`) and
`output/diagnostic_real_data.json` (from notebook 05, if real data ran).

**No new analysis in this notebook.** Only summary and interpretation.

---

In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../modules')
from constants import RM_THRESHOLD_RADM2, SIGMA_HALF, D_STAR

OUT = Path('../output')

# Load all available diagnostic summaries
summaries = {}
for f in OUT.glob('diagnostic*.json'):
    with open(f) as fp:
        summaries[f.stem] = json.load(fp)

if not summaries:
    print('No diagnostic output found.')
    print('Run notebook 05 first (or run bullet_engine.py directly).')
else:
    print(f'Loaded {len(summaries)} diagnostic result(s):')
    for name, s in summaries.items():
        print(f'\n  [{name}]')
        print(f'    Verdict:          {s.get("verdict")}')
        print(f'    DM model favoured: {s.get("dm_model_favoured")}')
        print(f'    Max |ΔRM|:         {s.get("max_abs_delta_rm")} rad/m²')
        print(f'    Threshold:         {s.get("threshold_radm2")} rad/m²')
        print(f'    Data source:       {s.get("data_source")}')

In [ ]:
# Summary table: predictions vs results
import sys
from constants import RM_THRESHOLD_RADM2

# Load prediction table from notebook 01
predictions = {
    'P_RM_NW':      {'wave': f'<{RM_THRESHOLD_RADM2}', 'particle': f'>{RM_THRESHOLD_RADM2}', 'unit': 'rad/m²'},
    'P_RM_SE':      {'wave': f'<{RM_THRESHOLD_RADM2}', 'particle': f'>{RM_THRESHOLD_RADM2}', 'unit': 'rad/m²'},
    'P_RM_κ_corr':  {'wave': '<0.2',  'particle': '>0.3', 'unit': 'Pearson r'},
    'P_RM_Σ_corr':  {'wave': '>0.8',  'particle': '<0.8', 'unit': 'Pearson r'},
}

print('RESULTS TABLE')
print('='*80)
print(f'{"Prediction":<20} {"Wave pred":<18} {"Particle pred":<18} {"Measured":<18} {"Status"}')
print('-'*80)

for name, pred in predictions.items():
    # Try to find measured value in diagnostic summaries
    measured = '—'
    status   = '(pending real data)'

    if 'diagnostic_real_data' in summaries:
        s = summaries['diagnostic_real_data']
        if name in ('P_RM_NW', 'P_RM_SE') and s.get('max_abs_delta_rm') is not None:
            measured = f'{s["max_abs_delta_rm"]:.1f} rad/m²'
            status   = s.get('verdict', '—')[:30]

    print(f'{name:<20} {pred["wave"]:<18} {pred["particle"]:<18} {measured:<18} {status}')

print('='*80)

In [ ]:
# Plot all available diagnostic RM profiles from saved .npz
npz_path = OUT / 'transect_profiles.npz'
if npz_path.exists():
    data = np.load(npz_path, allow_pickle=True)
    offset = data['offset_arcmin']
    channels = [k for k in data.files if k != 'offset_arcmin']

    fig, axes = plt.subplots(len(channels), 1, figsize=(10, 3*len(channels)), sharex=True)
    if len(channels) == 1: axes = [axes]

    for ax, ch in zip(axes, channels):
        arr = data[ch]
        if arr.ndim == 0 or arr is None: continue
        ax.plot(offset, arr, 'k', lw=0.8)
        ax.set_ylabel(ch)
        ax.set_title(ch)

    axes[-1].set_xlabel('Transect offset (arcmin)')
    plt.suptitle('All transect profiles (from saved .npz)')
    plt.tight_layout()
    plt.savefig(OUT / 'results_transect_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('transect_profiles.npz not found — run bullet_engine.py first')

## Interpretation

### If wave DM verdict (|ΔRM| < 5 rad/m²)

Dark matter contributes no Faraday rotation. The Bullet Cluster DM halos are
topologically transparent — consistent with the Ainulindale σ-face framework where
DM occupies the ZD boundary, not the CD plasma layer.

The mass that bends light is a *shape*, not a *substance*. The lensing is real;
the stuff is not.

Physics implication: the separation between X-ray gas and lensing mass is not
'stuff overtaking stuff' — it is 'plasma lagging behind a geometric attractor
(the L_(I|O) pathway).' The cluster merger is the system traversing the cardioid
geometry: gas at the loop, DM at the cusp.

### If particle DM verdict (|ΔRM| > 5 rad/m²)

DM carries charged particles and magnetic field. This does not disprove the
Ainulindale framework — it constrains which layer of the tower the DM halos
occupy. A positive result means the halos are above σ = ½ (CD), not at the
boundary itself.

Either result is scientifically valuable. The threshold is pre-registered.
The verdict stands.

### What this experiment does NOT resolve

- What generates the gravitational potential if DM has no electrons
- The mass budget (lensing still requires mass — the σ-face itself has inertia)
- Whether multiple ZD boundary layers can stack to produce observed κ profiles

These questions go to the D-P (physics) paper, not here.

---

## Data still pending

| Dataset | Status | Action |
|---|---|---|
| MeerKAT MGCLS Q/U cubes | SARAO registration required | `https://archive.sarao.ac.za/` proposal SSV-20180624-FC-01 |
| Planck SZ y-map | Downloading (11 GB tgz) | Extract when complete |
| JWST optical layers | Partially downloaded | Run `generate_layers.py` once complete |
| Published RM map (Shimwell+2014) | Digitize from paper figure | Alternative to SARAO cubes |
